In [14]:
import os
import pandas as pd
import sys
import gc
import pickle

project_root = os.path.abspath("..")   # lên 1 cấp: MIND-research

if project_root not in sys.path:
    sys.path.insert(0, project_root)

In [15]:
from src.core.Context import VectorContext
from src.represent.RepresentedVector import RepresentedVector

from bertopic import BERTopic
from sentence_transformers import SentenceTransformer

# Create Represented Vector List

In [16]:
# Semantic pretrained
semantic_model = SentenceTransformer("all-MiniLM-L6-v2")

# Topic model đã train
topic_model = BERTopic.load(project_root + "/models/primary/bertopic")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [17]:
context = VectorContext(
    os.path.join(project_root, "data", "primary", "test_set")
)
title_list = context.createTitleList()
titles = [item["title"] for item in title_list]

semantic_embeddings = semantic_model.encode(
    titles,
    show_progress_bar=True,
)

topics, topic_distributions = topic_model.transform(
    titles,
    embeddings=semantic_embeddings,
)

represented_vector_list = {
    item["news_id"]: {
        "semantic": semantic_embedding,
        "topic_distribution": topic_distribution,
    }
    for item, semantic_embedding, topic_distribution in zip(
        title_list,
        semantic_embeddings,
        topic_distributions,
    )
}

len(represented_vector_list)

Batches:   0%|          | 0/3780 [00:00<?, ?it/s]

2026-08-21 17:56:38,726 - BERTopic - Dimensionality - Reducing dimensionality of input embeddings.
2026-08-21 17:58:02,628 - BERTopic - Dimensionality - Completed ✓
2026-08-21 17:58:02,630 - BERTopic - Clustering - Approximating new points with `hdbscan_model`
2026-08-21 17:58:31,979 - BERTopic - Probabilities - Start calculation of probabilities with HDBSCAN
2026-08-21 20:04:28,742 - BERTopic - Probabilities - Completed ✓
2026-08-21 20:04:28,786 - BERTopic - Cluster - Completed ✓


120959

In [18]:
vectors_dir = os.path.join(project_root, "vectors", "primary")
os.makedirs(vectors_dir, exist_ok=True)

represent_vectors_path = os.path.join(vectors_dir, "represent_vectors.pkl")
with open(represent_vectors_path, "wb") as f:
    pickle.dump(represented_vector_list, f, protocol=pickle.HIGHEST_PROTOCOL)

print(f"Saved {len(represented_vector_list)} news vectors to {represent_vectors_path}")

Saved 120959 news vectors to d:\CDNC\MIND-research\vectors\primary\represent_vectors.pkl


In [19]:
del represented_vector_list
del semantic_embeddings
del topic_distributions
del titles
del title_list
del context
del semantic_model
del topic_model

gc.collect()

print("RAM cleanup completed.")

RAM cleanup completed.
